- **Saad Amir** — 20i-0650  
- **Talha Tanveer** — 20i-0438

- **Saad Amir** — 20i-0650  
- **Talha Tanveer** — 20i-0438.

- **Saad Amir** — 20i-0650  
- **Talha Tanveer** — 20i-0438

In [5]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import RandomHorizontalFlip, RandomRotation, RandomResizedCrop


In [6]:
class ColorizationDataset(Dataset):
    def __init__(self, root_dir, transform_gray=None, transform_color=None):
        """
        Args:
            root_dir (str): Directory containing 'gray' and 'color' subfolders.
            transform_gray (callable, optional): Transform to apply to grayscale images.
            transform_color (callable, optional): Transform to apply to color images.
        """
        self.gray_dir = os.path.join(root_dir, "gray")
        self.color_dir = os.path.join(root_dir, "color")
        
        # Load file names and ensure both have matching pairs
        self.gray_images = sorted(os.listdir(self.gray_dir))
        self.color_images = sorted(os.listdir(self.color_dir))
        assert len(self.gray_images) == len(self.color_images), "Mismatch in dataset sizes!"
        
        self.transform_gray = transform_gray
        self.transform_color = transform_color

    def __len__(self):
        return len(self.gray_images)

    def __getitem__(self, idx):
        # Load grayscale image
        gray_path = os.path.join(self.gray_dir, self.gray_images[idx])
        gray_image = Image.open(gray_path).convert("L")  # Ensure grayscale
        
        # Load color image
        color_path = os.path.join(self.color_dir, self.color_images[idx])
        color_image = Image.open(color_path).convert("RGB")  # Ensure RGB
        
        # Apply transforms if provided
        if self.transform_gray:
            gray_image = self.transform_gray(gray_image)
        if self.transform_color:
            color_image = self.transform_color(color_image)
        
        return gray_image, color_image



transform_gray = transforms.Compose([
    transforms.RandomResizedCrop((256, 256), scale=(0.8, 1.0)),  # Randomly crop and resize
    transforms.RandomHorizontalFlip(p=0.5),  # Random horizontal flip
    transforms.RandomRotation(10),  # Random rotation within 10 degrees
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize (mean, std) for grayscale
])

transform_color = transforms.Compose([
    transforms.RandomResizedCrop((256, 256), scale=(0.8, 1.0)),  # Randomly crop and resize
    transforms.RandomHorizontalFlip(p=0.5),  # Random horizontal flip
    transforms.RandomRotation(10),  # Random rotation within 10 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Adjust color properties
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize (mean, std) for RGB
])



In [ ]:
train_dataset = ColorizationDataset(
    root_dir="/kaggle/input/genai-colorify-dataset-version-1/train",
    transform_gray=transform_gray,
    transform_color=transform_color
)

test_dataset = ColorizationDataset(
    root_dir="/kaggle/input/genai-colorify-dataset-version-1/test",
    transform_gray=transform_gray,
    transform_color=transform_color
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


# Check a batch of data
for gray_batch, color_batch in train_loader:
    print("Gray batch shape:", gray_batch.shape)  # Should be [batch_size, 1, 256, 256]
    print("Color batch shape:", color_batch.shape)  # Should be [batch_size, 3, 256, 256]
    break

Gray batch shape: torch.Size([16, 1, 256, 256])
Color batch shape: torch.Size([16, 3, 256, 256])
